In [1]:
import pandas as pd

In [7]:
athletes = pd.read_csv('athlete_events.csv')

In [8]:
athletes.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [9]:
athletes = athletes[athletes["Season"]=="Summer"]

In [10]:
def team_summary(data):
    return pd.Series({
        'team': data.iloc[0,:]["NOC"],
        'country': data.iloc[-1,:]["Team"],
        'year': data.iloc[0,:]["Year"],
        'events': len(data['Event'].unique()),
        'athletes': data.shape[0],
        'age': data["Age"].mean(),
        'height': data['Height'].mean(),
        'weight': data['Weight'].mean(),
        'medals': sum(~pd.isnull(data["Medal"]))
    })

team = athletes.groupby(["NOC", "Year"]).apply(team_summary)

C:\Users\Khushi\AppData\Local\Temp\ipykernel_22940\3533186276.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  team = athletes.groupby(["NOC", "Year"]).apply(team_summary)


In [11]:
team = team.reset_index(drop=True)

In [12]:
team = team.dropna()

In [13]:
team

,team,country,year,events,athletes,age,height,weight,medals
3,AFG,Afghanistan,1960,13,16,23.312500,170.687500,69.437500,0
4,AFG,Afghanistan,1964,8,8,22.000000,161.000000,64.250000,0
5,AFG,Afghanistan,1968,5,5,23.200000,170.200000,70.000000,0
6,AFG,Afghanistan,1972,8,8,29.000000,168.333333,63.750000,0
7,AFG,Afghanistan,1980,11,11,23.636364,168.363636,63.181818,0
...,...,...,...,...,...,...,...,...,...
2805,ZIM,Zimbabwe,2000,19,26,24.961538,178.960000,71.080000,0
2806,ZIM,Zimbabwe,2004,11,14,25.071429,177.785714,70.500000,3
2807,ZIM,Zimbabwe,2008,15,16,26.062500,171.928571,63.714286,4
2808,ZIM,Zimbabwe,2012,8,9,27.333333,174.444444,65.222222,0


In [14]:
def prev_medals(data):
    data = data.sort_values("year", ascending=True)
    data["prev_medals"] = data["medals"].shift(1)
    data["prev_3_medals"] = data.rolling(3, closed="left", min_periods=1).mean()["medals"]
    return data

team = team.groupby(["team"]).apply(prev_medals)
team = team.reset_index(drop=True)
team = team[team["year"] > 1960]
team = team.round(1)

DataError: Cannot aggregate non-numeric type: object

In [15]:
team[team["team"]=="USA"]

,team,country,year,events,athletes,age,height,weight,medals
2673,USA,United States,1896,16,27,23.074074,179.875000,72.461538,20
2674,USA,A North American Team,1900,41,135,24.400000,179.047619,74.441176,63
2675,USA,United States,1904,95,1109,27.006764,175.779070,71.600000,394
2676,USA,United States,1906,28,81,24.913580,180.271186,71.525424,24
2677,USA,United States,1908,52,219,26.364486,179.476190,76.148936,65
2678,USA,United States,1912,68,364,28.132597,179.160550,73.622222,107
2679,USA,United States,1920,113,473,27.188960,176.754098,74.174825,194
2680,USA,United States,1924,108,459,26.595186,175.882562,71.008621,182
2681,USA,United States,1928,96,426,29.071429,178.948387,72.750000,88
2682,USA,United States,1932,122,743,36.005610,178.328767,71.701613,189


In [16]:
team

,team,country,year,events,athletes,age,height,weight,medals
3,AFG,Afghanistan,1960,13,16,23.312500,170.687500,69.437500,0
4,AFG,Afghanistan,1964,8,8,22.000000,161.000000,64.250000,0
5,AFG,Afghanistan,1968,5,5,23.200000,170.200000,70.000000,0
6,AFG,Afghanistan,1972,8,8,29.000000,168.333333,63.750000,0
7,AFG,Afghanistan,1980,11,11,23.636364,168.363636,63.181818,0
...,...,...,...,...,...,...,...,...,...
2805,ZIM,Zimbabwe,2000,19,26,24.961538,178.960000,71.080000,0
2806,ZIM,Zimbabwe,2004,11,14,25.071429,177.785714,70.500000,3
2807,ZIM,Zimbabwe,2008,15,16,26.062500,171.928571,63.714286,4
2808,ZIM,Zimbabwe,2012,8,9,27.333333,174.444444,65.222222,0


In [17]:
team.to_csv("teams.csv", index=False)